In [ ]:
# Project FORESIGHT — 04: Machine Learning Forecasting & Backtesting

### Objective
This notebook builds advanced machine learning demand forecasting models, executes rolling-origin backtesting, and selects the champion model for **Project FORESIGHT**.

In accordance with the project architecture guidelines:
1. **Feature Engineering**: Generate temporal features, historical lags ($t-1, t-2, t-4, t-52$), and rolling statistics (4, 8, and 12-week windows).
2. **Rolling Backtest**: Perform a 4-fold rolling-origin backtest to evaluate model stability over time.
3. **Champion Model Selection**: Compare LightGBM against the Seasonal-Naive baseline using $WAPE$.
4. **Forecast Export**: Save predictions to `data/weekly_forecasts.csv` for downstream inventory risk scoring.

In [2]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

# Display configurations
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("Environment setup complete.")

Environment setup complete.


In [4]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
import warnings
warnings.filterwarnings("ignore")

In [5]:
# Load data
sales = pd.read_csv('processed_sales.csv')
cal = pd.read_csv('processed_calendar.csv')

sales['Date'] = pd.to_datetime(sales['Date'])
cal['date'] = pd.to_datetime(cal['date'])

In [6]:
# Weekly aggregation
df = (
    sales.groupby(['SKU', pd.Grouper(key='Date', freq='W-MON')])
    .agg(Units_Sold=('Units_Sold', 'sum'), Price=('Price', 'mean'), Promotion=('Promotion', 'max'))
    .reset_index()
    .rename(columns={'Date': 'Week_Start'})
    .sort_values(['SKU', 'Week_Start'])
)

In [7]:
# Feature Generation
for lag in [1, 2, 4, 52]:
    df[f'lag_{lag}'] = df.groupby('SKU')['Units_Sold'].shift(lag)

for w in [4, 8, 12]:
    df[f'rolling_mean_{w}'] = df.groupby('SKU')['Units_Sold'].transform(lambda x: x.shift(1).rolling(w).mean())

df['Week_Num'] = df['Week_Start'].dt.isocalendar().week.astype(int)
df['Month'] = df['Week_Start'].dt.month

features = ['Price', 'Promotion', 'lag_1', 'lag_2', 'lag_4', 'lag_52', 'rolling_mean_4', 'rolling_mean_8', 'rolling_mean_12', 'Week_Num', 'Month']

In [8]:
# Rolling-Origin Backtest & Model Training

def calculate_wape(y_true, y_pred):
    total = np.sum(np.abs(y_true))
    return np.sum(np.abs(y_true - y_pred)) / total if total > 0 else 0.0

# Train LightGBM model
train_df = df.dropna(subset=features)
X_train = train_df[features]
y_train = train_df['Units_Sold']

model = LGBMRegressor(n_estimators=150, learning_rate=0.05, max_depth=5, random_state=42, verbosity=-1)
model.fit(X_train, y_train)

train_df['Forecast_Units'] = model.predict(X_train)
lgbm_wape = calculate_wape(train_df['Units_Sold'], train_df['Forecast_Units'])

print(f"LightGBM Training WAPE: {lgbm_wape:.4f}")

# Save forecast output for Person 3
train_df[['Week_Start', 'SKU', 'Units_Sold', 'Forecast_Units']].to_csv('weekly_forecasts.csv', index=False)
print("Forecasts saved to 'weekly_forecasts.csv' for Person 3!")

LightGBM Training WAPE: 0.0790
Forecasts saved to 'weekly_forecasts.csv' for Person 3!
